In [56]:
%pip install pyiceberg pyarrow pandas requests
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pyiceberg.catalog import load_catalog
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import os
from pathlib import Path
from sqlalchemy import create_engine, text
load_dotenv(Path(".dev.env"))
API = os.getenv("API")

In [2]:
API = os.getenv("API")

"""https://places.foursquare.com/dataset/OS%20Categories/code"""


# Charger le catalogue
catalog = load_catalog(
    "default",
    **{
        "warehouse": "places",
        "uri": "https://catalog.h3-hub.foursquare.com/iceberg",
        "token": API,
        "header.content-type": "application/vnd.api+json",
        "rest-metrics-reporting-enabled": "false",
    },
)

selected_columns = ["category_id", "category_name"]
table = catalog.load_table('datasets.categories_os')
df_cat = table.scan(row_filter =("(category_name IS NOT NULL)")).to_pandas()

print(df_cat["category_name"].size)
print(table.current_snapshot())


1279
Operation.APPEND: id=7593517863721436250, schema_id=1


In [3]:

df_cat = df_cat.drop_duplicates(subset=["category_name"])
print(df_cat["category_name"].size)


# on a un duplicated à la valeur "Restaurant", donc 2 id différents pour la meme categorie
# on choisit de drop les duplicated pour alléger la base, il y a déjà plein de sortes de restaurant différents
# on a 1277 types de categories dont 266 pour restaurant
#Le premier "Restaurant" sera gardé

print(df_cat[df_cat["category_name"]=="Restaurant"])


1278
                  category_id  category_level category_name  \
384  4d4b7105d754a06374d81259               2    Restaurant   

                       category_label        level1_category_id  \
384  Dining and Drinking > Restaurant  63be6904847c3692a84b9bb5   

    level1_category_name        level2_category_id level2_category_name  \
384  Dining and Drinking  4d4b7105d754a06374d81259           Restaurant   

    level3_category_id level3_category_name level4_category_id  \
384               None                 None               None   

    level4_category_name level5_category_id level5_category_name  \
384                 None               None                 None   

    level6_category_id level6_category_name  
384               None                 None  


In [10]:
from pyiceberg.catalog import load_catalog

API=os.getenv("API")

# Charger le catalogue
catalog = load_catalog(
    "default",
    **{
        "warehouse": "places",
        "uri": "https://catalog.h3-hub.foursquare.com/iceberg",
        "token": API,
        "header.content-type": "application/vnd.api+json",
        "rest-metrics-reporting-enabled": "false",
    },
)

selected_columns = [
    "fsq_place_id", "name", "latitude", "longitude",
    "address", "postcode", "tel", "website", "fsq_category_ids",
    "date_refreshed"
]

print("Chargement de la table...")
table = catalog.load_table('datasets.places_os')
print("Table chargée ! Transformation en DataFrame...")
print(table.current_snapshot())
df_plc = table.scan(
        #selected_fields=selected_columns,
        row_filter=(
        "(latitude IS NOT NULL) AND "
        "(longitude IS NOT NULL) AND "
        "(country = 'FR') AND "
        #"(locality = 'Paris' OR locality = 'PARIS') AND "
        "(postcode LIKE '75%') AND"
        "(region IS NOT NULL) AND "
        "(address IS NOT NULL)"
                
    )#, limit=1000
).to_pandas()

Chargement de la table...
Table chargée ! Transformation en DataFrame...
Operation.OVERWRITE: id=3317796563964136411, parent_id=8863925977174862667, schema_id=90


In [4]:

from pyiceberg.catalog import load_catalog

API=os.getenv("API")

# Charger le catalogue
catalog = load_catalog(
    "default",
    **{
        "warehouse": "places",
        "uri": "https://catalog.h3-hub.foursquare.com/iceberg",
        "token": API,
        "header.content-type": "application/vnd.api+json",
        "rest-metrics-reporting-enabled": "false",
    },
)
table = catalog.load_table('datasets.places_os')
selected_columns = [
    "fsq_place_id", "name", "latitude", "longitude",
    "address", "postcode", "tel", "website", "fsq_category_ids",
    "date_refreshed", 
]

df = table.scan(

    row_filter="fsq_place_id = '4e29ebc28877b69d49cfa225'"
).to_pandas()

print(df.T)

                                                                     0
fsq_place_id                                  4e29ebc28877b69d49cfa225
name                                            Les Freres De Rodellec
latitude                                                     48.879839
longitude                                                     2.326867
address                                             47 rue d'Amsterdam
locality                                                         Paris
region                                                   Île-de-France
postcode                                                         75009
admin_region                                             Île-de-France
post_town                                                         None
po_box                                                            None
country                                                             FR
date_created                                                2011-07-22
date_r

In [6]:
old = table.metadata.snapshots[-5].snapshot_id
df = table.scan(
    snapshot_id=old,

    row_filter="fsq_place_id = '4e29ebc28877b69d49cfa225'"
).to_pandas()

print(df)

               fsq_place_id                    name   latitude  longitude  \
0  4e29ebc28877b69d49cfa225  Les Freres De Rodellec  48.879839   2.326867   

              address locality         region postcode   admin_region  \
0  47 rue d'Amsterdam   Parigi  Île-de-France    75009  Île-de-France   

  post_town  ... email facebook_id instagram twitter  \
0      None  ...  None         NaN      None    None   

             fsq_category_ids  \
0  [4bf58dd8d48988d141941735]   

                                 fsq_category_labels  \
0  [Community and Government > Education > Colleg...   

                                      placemaker_url unresolved_flags  \
0  https://foursquare.com/placemakers/review-plac...             None   

                                                geom  \
0  b'\x00\x00\x00\x00\x01@\x02\x9dli\x01;r@Hp\x9e...   

                                                bbox  
0  {'xmin': 2.3268669322491684, 'ymin': 48.879838...  

[1 rows x 27 columns]


In [7]:
df.head()

,fsq_place_id,name,latitude,longitude,address,locality,region,postcode,admin_region,post_town,...,email,facebook_id,instagram,twitter,fsq_category_ids,fsq_category_labels,placemaker_url,unresolved_flags,geom,bbox
0,4e29ebc28877b69d49cfa225,Les Freres De Rodellec,48.879839,2.326867,47 rue d'Amsterdam,Parigi,Île-de-France,75009,Île-de-France,None,...,None,NaN,None,None,[4bf58dd8d48988d141941735],[Community and Government > Education > Colleg...,https://foursquare.com/placemakers/review-plac...,None,b'\x00\x00\x00\x00\x01@\x02\x9dli\x01;r@Hp\x9e...,"{'xmin': 2.3268669322491684, 'ymin': 48.879838..."


In [70]:
# il y a dans la colonne fsq_category_ids certains qui ont un nan. On va supprimer ces lignes
df_plc = df_plc.dropna(subset=["fsq_category_ids"])

# on supprime les duplicates qui ont le meme id, name, adress, lat et long
df_plc = df_plc.drop_duplicates(subset=["fsq_place_id","name","address", "latitude", "longitude"])
print(df_plc["fsq_place_id"].size)

# il y a bcp de duplicates par le nom, c'est normal car ça peut etre par exemple un nom de magasin(Darty) ou une station de recharge ( Electric Charging Station)
duplicate = df_plc[df_plc.duplicated(subset='name')]


906


In [71]:
# Collecter toutes les catégories présentes dans df_plc, avoir que celles qui ont un poi associé

all_categories = set()
for categories in df_plc['fsq_category_ids']:
    if isinstance(categories, (list, np.ndarray)):
        all_categories.update(categories)
    elif isinstance(categories, str):
        all_categories.add(categories)

print(all_categories)
df_cat = df_cat[df_cat['category_id'].isin(all_categories)]
print(df_cat['category_id'].size)

{'4f04afc02fb6e1c99f3db0bc', '4bf58dd8d48988d175941735', '5032885091d4c4b30a586d66', '5283c7b4e4b094cb91ec88d7', '4d4b7105d754a06374d81259', '52e81612bcbc57f1066b7a22', '52f2ab2ebcbc57f1066b8b4f', '63be6904847c3692a84b9bd1', '4deefb944765f83613cdba6e', '4bf58dd8d48988d145941735', '5745c2e4498e11e7bccabdbd', '4bf58dd8d48988d1f6941735', '4bf58dd8d48988d1ff931735', '4bf58dd8d48988d142941735', '4bf58dd8d48988d10c951735', '4bf58dd8d48988d101951735', '63be6904847c3692a84b9b46', '4bf58dd8d48988d1f8941735', '63be6904847c3692a84b9b59', '4bf58dd8d48988d110951735', '4bf58dd8d48988d18f941735', '63be6904847c3692a84b9b3f', '522e32fae4b09b556e370f19', '4bf58dd8d48988d1a0941735', '4bf58dd8d48988d164941735', '4bf58dd8d48988d10c941735', '4eb1daf44b900d56c88a4600', '4d954afda243a5684865b473', '4bf58dd8d48988d1fd931735', '4bf58dd8d48988d13d941735', '63be6904847c3692a84b9b24', '56aa371be4b08b9a8d573520', '4bf58dd8d48988d1a2941735', '63be6904847c3692a84b9b3d', '4f04b08c2fb6e1c99f3db0bd', '4bf58dd8d48988d105

In [72]:
df_plc.rename(
    columns={"fsq_place_id": "idFsq", "latitude": "latitudePoi", "longitude":"longitudePoi"}, inplace=True
)
data_to_insert = df_plc.drop(columns=["fsq_category_ids"]).to_dict(orient="records")
print(data_to_insert)

[{'idFsq': '4bd93892e914a5936ffe55fa', 'name': 'Surcouf', 'latitudePoi': 44.83153700518116, 'longitudePoi': -0.6589617481623053, 'address': '139 Ave. Daumesnil', 'postcode': '75012', 'tel': None, 'website': None}, {'idFsq': '59ccd3c30d2be74d9e888817', 'name': 'LA HALLE AUX CHAUSSURES PARIS CLICHY', 'latitudePoi': 44.839161, 'longitudePoi': -0.573654, 'address': '29 avenue De Clichy', 'postcode': '75017', 'tel': '01 53 04 34 41', 'website': 'https://www.lahalle.com/magasins/paris/paris/la-halle-aux-chaussures-paris-clichy-040785'}, {'idFsq': '5a9699cae55d8b2f3e912129', 'name': 'Self Défense Entreprise', 'latitudePoi': 44.83464525111451, 'longitudePoi': -0.5833291120002441, 'address': '108 bis Auguste Blanqui', 'postcode': '75013', 'tel': '06 85 87 14 15', 'website': 'https://www.formationprosecurite.com'}, {'idFsq': 'f0184a59726c4a889d17c578', 'name': 'Joseph a un doute', 'latitudePoi': 44.854495861436114, 'longitudePoi': -0.5658237476848426, 'address': '231 rue Saint Honoré', 'postcode

In [77]:
# il y a des categories absentes dans le catalogue categorie mais presentes dans la colonne fsq_categories_ids
# un filtre supplémentaire de ces categories doit etre ajoute

valid_categories = set(df_cat["category_id"])

missing = set(df_rel["category_id"]) - valid_categories

if missing:
    print(
        f"Attention : {len(missing)} catégories absentes du catalogue"
    )

df_rel = df_rel[df_rel["category_id"].isin(valid_categories)]

KeyError: 'category_id'

In [ ]:

# Charger le catalogue DELTAS
catalog = load_catalog(
    "default",
    **{
    "warehouse": "places",
    "uri": "https://catalog.h3-hub.foursquare.com/iceberg",
    "token": API,
    "header.content-type": "application/vnd.api+json",
    "rest-metrics-reporting-enabled": "false",
},
)


table = catalog.load_table('datasets.deltas_os')


df_delta = table.scan( 
).to_pandas()

print(table.current_snapshot())

Chargement de la table...
Table chargée ! Transformation en DataFrame...
Operation.APPEND: id=2973393988795176882, schema_id=1


In [ ]:
table.current_snapshot().snapshot_id # avoir le id du snapshot courant pour le versionning

2973393988795176882